# Governance Layer — Full Experimental Reproducibility

This notebook reproduces all results from the Governance Layer framework.

**What it does:**
1. Verifies 12 formal predictions from Chapters 2-4 (prove.py)
2. Trains a PPO agent under Parliament governance (10×10 GridWorld)
3. Trains a PPO agent WITHOUT governance (baseline)
4. Runs benchmark with multiple seeds
5. Exports results to CSV and commits to GitHub

**Runtime:** ~15 minutes with T4 GPU

In [ ]:
# Install dependencies
!pip install -q torch stable-baselines3 gymnasium neo4j streamlit altair pandas numpy 2>&1 | tail -5
print("Dependencies installed")

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/xcoder-es/governance-layer.git"
REPO_DIR = "/content/governance-layer"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Always pull latest on every run
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Working in {os.getcwd()}")

## 1. Prove.py — Formal Prediction Verification

Runs all 12 predictions from the book. Each maps a mathematical claim to an executable test.

In [ ]:
from src.governance.prove.runner import run_all, print_summary

results = run_all()
print_summary(results)

passed = sum(1 for r in results if r.passed)
total = len(results)
print(f"\nVerification: {passed}/{total} predictions PASS")
assert passed == total, f"{total - passed} predictions FAILED — investigate before proceeding"

## 2. Train PPO under Governance

Trains a PPO agent in the 10×10 GridWorld with the Parliament filtering actions.
Agent proposes direction → Parliament evaluates → if approved, action executes.

In [ ]:
from src.governance.experiments.rl_train import train_ppo

print("Training governance agent (100k timesteps)...")
gov_result = train_ppo(
    mode="governance",
    total_timesteps=100_000,
    size=10,
    seed=42,
    log_dir="results",
    eval_episodes=10,
)
e = gov_result["eval"]
print(f"  Avg reward: {e['avg_reward']:.2f} +/- {e['std_reward']:.2f}")
print(f"  Avg violations: {e['avg_violations']:.2f}")
print(f"  Avg apples: {e['avg_apples']:.1f}")

## 3. Train PPO without Governance (Baseline)

Same environment, same PPO hyperparameters. Actions go directly to the grid.

In [ ]:
print("Training no-governance agent (100k timesteps)...")
nogov_result = train_ppo(
    mode="no_governance",
    total_timesteps=100_000,
    size=10,
    seed=42,
    log_dir="results",
    eval_episodes=10,
)
e2 = nogov_result["eval"]
print(f"  Avg reward: {e2['avg_reward']:.2f} +/- {e2['std_reward']:.2f}")
print(f"  Avg violations: {e2['avg_violations']:.2f}")
print(f"  Avg apples: {e2['avg_apples']:.1f}")

## 4. Comparison

How does governance affect reward and violations?

In [ ]:
import numpy as np

print("=" * 60)
print("  GOVERNANCE vs NO GOVERNANCE")
print("=" * 60)

g_reward = gov_result["eval"]["avg_reward"]
g_viol = gov_result["eval"]["avg_violations"]
g_apples = gov_result["eval"]["avg_apples"]

n_reward = nogov_result["eval"]["avg_reward"]
n_viol = nogov_result["eval"]["avg_violations"]
n_apples = nogov_result["eval"]["avg_apples"]

print(f"  {'Metric':<20} {'Governance':<20} {'No Governance':<20}")
print(f"  {'-'*60}")
print(f"  {'Avg Reward':<20} {g_reward:<20.2f} {n_reward:<20.2f}")
print(f"  {'Avg Violations':<20} {g_viol:<20.2f} {n_viol:<20.2f}")
print(f"  {'Avg Apples':<20} {g_apples:<20.1f} {n_apples:<20.1f}")

print(f"\n  Hypothesis: Governance reduces violations while maintaining reward.")
if g_viol < n_viol:
    print(f"  RESULT: SUPPORTED (governance violations={g_viol:.1f} < no_governance={n_viol:.1f})")
else:
    print(f"  RESULT: NOT SUPPORTED — investigate")

## 5. Multi-Seed Benchmark

Runs 3 seeds for statistical confidence. Takes ~10 minutes.

In [ ]:
from src.governance.experiments.rl_train import benchmark

print("Running 3-seed benchmark (this takes ~10 minutes)...")
bench_results = benchmark(
    total_timesteps=100_000,
    size=10,
    seeds=[42, 43, 44],
    log_dir="results",
    eval_episodes=10,
)

print("=" * 60)
print("  BENCHMARK RESULTS")
print("=" * 60)
for mode, data in bench_results.items():
    print(f"  {mode}:")
    print(f"    Reward:     {data['avg_reward']:.2f} +/- {data['std_reward']:.2f}")
    print(f"    Violations: {data['avg_violations']:.2f}")
    print()

## 6. Export Results

Downloads all result files for use with the Streamlit dashboard.

In [ ]:
import shutil
from google.colab import files

!ls -la results/

print("\nDownloading results...")
for fname in os.listdir("results"):
    if fname.endswith(".json") or fname.endswith(".jsonl") or fname.endswith(".zip"):
        path = os.path.join("results", fname)
        files.download(path)

print("\nDone! Place these files in your local results/ directory and run:")
print("  streamlit run src/governance/dashboard/app.py")

## 7. Push Results to GitHub (Optional)

Uncomment and fill in your PAT to commit results directly to the repo.

In [ ]:
# from google.colab import userdata
# GITHUB_PAT = userdata.get('GITHUB_PAT')
# 
# if GITHUB_PAT:
#     !git config user.email "colab@reproducibility.ai"
#     !git config user.name "Colab Reproducibility Bot"
#     !git remote set-url origin https://{GITHUB_PAT}@github.com/xcoder-es/governance-layer.git
#     !git add results/
#     !git commit -m "colab: auto-committed benchmark results"
#     !git push
#     print("Results pushed to GitHub.")
# else:
#     print("No PAT found. Set GITHUB_PAT in Colab secrets (key icon in sidebar).")

## Summary

| Step | Status | Output |
|---|---|---|
| Prove.py (12 predictions) | ✅ | results/prove_results.json |
| Governance PPO (100k steps) | ✅ | results/results_governance.json, results/ppo_governance.zip |
| No-Governance PPO (100k steps) | ✅ | results/results_no_governance.json, results/ppo_no_governance.zip |
| Multi-seed benchmark | ✅ | results/benchmark_results.json |
| Live replay logs | ✅ | results/live_*.jsonl |

**Next steps:**
1. Download all files from `results/`
2. Place them in your local `results/` directory
3. Run: `streamlit run src/governance/dashboard/app.py`
4. Or if you have Neo4j: `streamlit run src/governance/dashboard/app.py -- --neo4j`